# OCR + Vision Pipeline

Advanced PDF extraction pipeline that handles:
- **Native text** extraction (fast, for digital PDFs)
- **Vision OCR** for scanned pages (using HuggingFace Gemma 4 Vision)
- **Chart/Figure description** using Vision LLM
- **Structured Markdown** output ready for RAG ingestion

Pipeline: `PDF → PyMuPDF → Vision OCR (scanned pages) → Image Description → Markdown → Chunk → Embed → Store`

---

In [3]:
import sys
import os
from pathlib import Path
from _setup import PROJECT_ROOT

print(f'Project root: {PROJECT_ROOT}')
print(f"HUGGINGFACE_Token set: {'[OK]' if os.getenv('HUGGINGFACE_Token') else '[MISSING] (required for Vision OCR)'}")

Project root: C:\Users\lenovo\Desktop\Pfe 2026\financial-advisor
HUGGINGFACE_Token set: [OK]


## 1. Basic PDF Text Extraction (No OCR)

Fast extraction using PyMuPDF for digitally-created PDFs.

In [4]:
import fitz  # PyMuPDF

# Point to any PDF in the project
PDF_PATH = str(PROJECT_ROOT / "data" / "docs" / "88-Article.pdf")

doc = fitz.open(PDF_PATH)
print(f"PDF: {Path(PDF_PATH).name}")
print(f"Pages: {len(doc)}")
print(f"Title: {doc.metadata.get('title', 'Unknown')}")
print(f"Author: {doc.metadata.get('author', 'Unknown')}")

# Extract text from first page
page = doc[0]
text = page.get_text("text")
print(f"\n--- Page 1 Text ({len(text)} chars) ---")
print(text[:500])
doc.close()

PDF: 88-Article.pdf
Pages: 23
Title: 
Author: Windows User

--- Page 1 Text (1186 chars) ---
Revue du Contrôle de la Comptabilité et de l’Audit  
ISSN: 2550-469X 
Numéro 8 : Mars 2019 
 
Hosting by COPERNICUS & CITEFACTOR                www.revuecca.com  
Page 152 
 
 
Evolution du système financier public marocain vers le 
Management de la Performance  
 
Evolution of the Moroccan public financial system towards the 
Performance Management 
 
Mohammed Achraf Nafzaoui 
Enseignant Chercheur à l'ENCG Kenitra 
Docteur HDR en Sciences de Gestion 
Laboratoire de Recherche en Sciences de Gest


## 2. Detect Scanned vs Digital Pages

In [5]:
doc = fitz.open(PDF_PATH)

print(f"Page Analysis for: {Path(PDF_PATH).name}")
print(f"{'Page':<6} {'Text Chars':<12} {'Images':<8} {'Type'}")
print("-" * 50)

for i in range(min(len(doc), 10)):  # First 10 pages
    page = doc[i]
    text = page.get_text("text").strip()
    images = page.get_images(full=True)
    
    # If very little text, it's likely scanned
    page_type = "SCANNED" if len(text) < 50 else "DIGITAL"
    print(f"{i+1:<6} {len(text):<12} {len(images):<8} {page_type}")

doc.close()

Page Analysis for: 88-Article.pdf
Page   Text Chars   Images   Type
--------------------------------------------------
1      1184         1        DIGITAL
2      1918         1        DIGITAL
3      2735         1        DIGITAL
4      2659         1        DIGITAL
5      2536         1        DIGITAL
6      2543         1        DIGITAL
7      2481         1        DIGITAL
8      2480         1        DIGITAL
9      2610         1        DIGITAL
10     2640         1        DIGITAL


## 3. Run Full OCR Pipeline

Uses Vision AI for scanned pages and chart descriptions.

Requires `HUGGINGFACE_Token` in `.env`

In [6]:
from OCR_Processing.pdf_extractor import AdvancedPDFPipeline

# Initialize the pipeline
pipeline = AdvancedPDFPipeline()

print(f"Processing: {Path(PDF_PATH).name}")
print("This may take a few minutes for large PDFs with scanned pages...\n")

# Run the pipeline
output_path = pipeline.process_pdf(PDF_PATH)

print(f"\n[OK] Output saved to: {output_path}")

2026-07-14 14:30:46,901 - INFO - Starting advanced extraction on: 88-Article.pdf
2026-07-14 14:30:46,906 - INFO - Processing page 1/23
2026-07-14 14:30:46,929 - INFO - Page 1: text only.
2026-07-14 14:30:46,931 - INFO - Processing page 2/23
2026-07-14 14:30:46,937 - INFO - Page 2: text only.
2026-07-14 14:30:46,938 - INFO - Processing page 3/23
2026-07-14 14:30:46,948 - INFO - Page 3: text only.
2026-07-14 14:30:46,949 - INFO - Processing page 4/23
2026-07-14 14:30:46,953 - INFO - Page 4: text only.
2026-07-14 14:30:46,955 - INFO - Processing page 5/23
2026-07-14 14:30:46,964 - INFO - Page 5: text only.
2026-07-14 14:30:46,965 - INFO - Processing page 6/23
2026-07-14 14:30:46,969 - INFO - Page 6: text only.
2026-07-14 14:30:46,973 - INFO - Processing page 7/23
2026-07-14 14:30:46,980 - INFO - Page 7: text only.
2026-07-14 14:30:46,981 - INFO - Processing page 8/23
2026-07-14 14:30:46,990 - INFO - Page 8: text only.
2026-07-14 14:30:46,993 - INFO - Processing page 9/23
2026-07-14 14:30:

Processing: 88-Article.pdf
This may take a few minutes for large PDFs with scanned pages...



2026-07-14 14:30:47,098 - INFO - Page 20: text only.
2026-07-14 14:30:47,101 - INFO - Processing page 21/23
2026-07-14 14:30:47,109 - INFO - Page 21: text only.
2026-07-14 14:30:47,118 - INFO - Processing page 22/23
2026-07-14 14:30:47,125 - INFO - Page 22: text only.
2026-07-14 14:30:47,127 - INFO - Processing page 23/23
2026-07-14 14:30:47,138 - INFO - Page 23: text only.
2026-07-14 14:30:47,147 - INFO - Extraction complete! Saved structured Markdown to: OCR_Processing\output\88-Article_extracted.md



[OK] Output saved to: OCR_Processing\output\88-Article_extracted.md


In [7]:
# Read and display the extracted Markdown
md_content = Path(output_path).read_text(encoding="utf-8")

print(f"Extracted Markdown: {len(md_content)} chars\n")
print("--- First 2000 chars ---")
print(md_content[:2000])

Extracted Markdown: 54924 chars

--- First 2000 chars ---
# Document Metadata: 88-Article.pdf

- **Pages**: 23
- **Title**: 
- **Author**: Windows User

---

## Page 1

Revue du Contrôle de la Comptabilité et de l’Audit  
ISSN: 2550-469X 
Numéro 8 : Mars 2019 
 
Hosting by COPERNICUS & CITEFACTOR                www.revuecca.com  
Page 152 
 
 
Evolution du système financier public marocain vers le 
Management de la Performance  
 
Evolution of the Moroccan public financial system towards the 
Performance Management 
 
Mohammed Achraf Nafzaoui 
Enseignant Chercheur à l'ENCG Kenitra 
Docteur HDR en Sciences de Gestion 
Laboratoire de Recherche en Sciences de Gestion des Organisations (LRSGO) 
ENCG KENITRA – Université Ibn Tofail 
Adresse mail : achrafnafzaoui@yahoo.fr 
 
Soukayna FERDOUSSI 
Doctorante en Sciences Economiques et Gestion à l'ENCG Kenitra 
Laboratoire de Recherche en Sciences de Gestion des Organisations (LRSGO) 
ENCG KENITRA – Université Ibn Tofail 
Adresse mail : ferdouss

## 4. Ingest OCR Output into ChromaDB

Take the extracted Markdown and run it through the ingestion pipeline.

In [8]:
from ingestion import IngestionPipeline
from ingestion.chunkers.table_aware import TableAwareChunker

# Use the table-aware chunker for OCR output (preserves table structures)
ingestion_pipeline = IngestionPipeline()

# Create records from the markdown
records = [(md_content, {
    "source": Path(PDF_PATH).name,
    "doc_type": "financial_report",
    "source_file": Path(PDF_PATH).name,
    "extraction_method": "ocr_vision",
    "ocr": "true",
})]

# Use TableAwareChunker for better preservation of financial tables
chunker = TableAwareChunker(chunk_size=800, chunk_overlap=100)

result = ingestion_pipeline.ingest_records(records, chunker=chunker)

print(f"\n[OK] Ingestion Result:")
print(f"  Documents: {result['documents']}")
print(f"  Chunks:    {result['chunks']}")
print(f"  Stored:    {result['stored']}")

2026-07-14 14:31:36,625 - INFO - TensorFlow version 2.16.1 available.
2026-07-14 14:31:36,631 - INFO - JAX version 0.5.3 available.
2026-07-14 14:31:38,597 - INFO - Embedder: loading model 'BAAI/bge-m3'
2026-07-14 14:31:38,607 - INFO - Use pytorch device_name: cpu
2026-07-14 14:31:38,608 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3
2026-07-14 14:31:39,412 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 14:31:39,437 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
2026-07-14 14:31:39,578 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 14:31:39,580 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits an

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-07-14 14:31:41,618 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/refs%2Fpr%2F130 "HTTP/1.1 200 OK"
2026-07-14 14:31:41,730 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 14:31:41,762 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/refs%2Fpr%2F130/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-07-14 14:31:41,764 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config.json "HTTP/1.1 200 OK"
2026-07-14 14:31:41,905 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/refs%2Fpr%2F130/model.safetensors "HTTP/1.1 302 Found"
2026-07-14 14:31:41,910 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 14:31:41,935 - INFO - HTTP Request: HEAD https://h


[OK] Ingestion Result:
  Documents: 1
  Chunks:    164
  Stored:    164


## 5. Compare: Basic Loader vs OCR Pipeline

In [9]:
from ingestion.loaders.pdf_loader import PDFLoader

# Basic extraction
basic_loader = PDFLoader()
basic_records = basic_loader.load(PDF_PATH)

basic_text = basic_records[0][0] if basic_records else ""

print("Comparison: Basic Loader vs OCR Pipeline")
print("=" * 50)
print(f"  Basic Loader:  {len(basic_text):,} chars extracted")
print(f"  OCR Pipeline:  {len(md_content):,} chars extracted")
print(f"  Improvement:   {((len(md_content) - len(basic_text)) / max(len(basic_text), 1) * 100):.1f}%")
print(f"\n  OCR adds: structured headings, table preservation, chart descriptions")

2026-07-14 14:35:59,633 - INFO - PDFLoader: loaded 23 pages from '88-Article.pdf'


Comparison: Basic Loader vs OCR Pipeline
  Basic Loader:  1,124 chars extracted
  OCR Pipeline:  54,924 chars extracted
  Improvement:   4786.5%

  OCR adds: structured headings, table preservation, chart descriptions
